# Traducción del dataset `eu_ai_act_flagged.csv` → español

Este notebook traduce las columnas de texto libre del dataset al español y añade la columna `etiqueta`
derivada de `severity`, dejando el archivo listo para el pipeline de `classifier_2`.

**Columnas traducidas:** `text`, `explanation`, `context`  
**Columnas mantenidas:** `violation`, `category`, `severity`, `articles`, `split`, `ambiguity`  
**Columna añadida:** `etiqueta` (`severity` → etiqueta del AI Act en español)

**Salida:** `data/eu_ai_act_flagged_es.csv`

## 0. Instalación de dependencias

In [2]:
import subprocess
import sys
def _pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

try:
    from deep_translator import GoogleTranslator
    print('✓ deep-translator ya instalado')
except ImportError:
    print('Instalando deep-translator…')
    _pip('deep-translator')
    from deep_translator import GoogleTranslator

try:
    from tqdm import tqdm
    print('✓ tqdm ya instalado')
except ImportError:
    _pip('tqdm')
    from tqdm import tqdm

✓ deep-translator ya instalado
✓ tqdm ya instalado


## 1. Carga del dataset original

In [3]:
import pandas as pd

INPUT_PATH  = '../data/eu_ai_act_flagged.csv'   # dataset original en classifier/data/
OUTPUT_PATH = 'datasets/eu_ai_act_flagged_es.csv'   # traducción en classifier_2/data/

df = pd.read_csv(INPUT_PATH)

print(f'Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas')
print(f'Columnas: {df.columns.tolist()}')
print()
print('Ejemplo de texto original (text):')
print(df['text'].iloc[0])

Dataset cargado: 300 filas, 9 columnas
Columnas: ['text', 'violation', 'category', 'severity', 'articles', 'explanation', 'context', 'split', 'ambiguity']

Ejemplo de texto original (text):
A smart city deploys an AI system for traffic light optimization that includes a public dashboard explaining the system's purpose, data sources, and decision-making logic, with real-time updates on how AI adjustments impact traffic flow.


## 2. Mapeo `severity` → `etiqueta`

| severity | etiqueta |
|---|---|
| `none` | `riesgo_minimo` |
| `borderline` | `riesgo_limitado` |
| `high` | `alto_riesgo` |
| `critical` | `inaceptable` |

In [4]:
SEVERITY_MAP = {
    'none':       'riesgo_minimo',
    'borderline': 'riesgo_limitado',
    'high':       'alto_riesgo',
    'critical':   'inaceptable',
}

df['etiqueta'] = df['severity'].map(SEVERITY_MAP)

# Verificar que no quedan nulos (valores de severity no contemplados)
nulos = df['etiqueta'].isna().sum()
if nulos:
    print(f'⚠  {nulos} valores de severity sin mapear:')
    print(df[df['etiqueta'].isna()]['severity'].value_counts())
else:
    print('✓ Mapeo completado sin valores nulos')

print()
print('Distribución de etiquetas:')
print(df['etiqueta'].value_counts())

✓ Mapeo completado sin valores nulos

Distribución de etiquetas:
etiqueta
inaceptable        131
riesgo_minimo      103
alto_riesgo         46
riesgo_limitado     20
Name: count, dtype: int64


## 3. Traducción de columnas de texto

Se traducen `text`, `explanation` y `context` con Google Translate (vía `deep-translator`).
Se guardan puntos de control cada 50 filas para no perder trabajo si hay un error de red.

In [5]:
import time

translator = GoogleTranslator(source='en', target='es')

def traducir_seguro(texto: str, reintentos: int = 3) -> str:
    """Traduce un texto con reintentos en caso de error de red."""
    if not isinstance(texto, str) or texto.strip() == '':
        return texto
    for intento in range(reintentos):
        try:
            return translator.translate(texto)
        except Exception as e:
            if intento < reintentos - 1:
                time.sleep(2 ** intento)   # espera exponencial: 1s, 2s
            else:
                print(f'  ⚠ Error al traducir (devuelvo original): {e}')
                return texto

COLUMNAS_A_TRADUCIR = ['text', 'explanation', 'context']
DELAY_ENTRE_FILAS   = 0.15   # segundos entre peticiones para no saturar la API
CHECKPOINT_CADA     = 50     # guardar CSV parcial cada N filas

df_es = df.copy()

for col in COLUMNAS_A_TRADUCIR:
    print(f'\nTraduciendo columna "{col}"…')
    traducciones = []
    for i, texto in enumerate(tqdm(df_es[col], desc=col)):
        traducciones.append(traducir_seguro(texto))
        time.sleep(DELAY_ENTRE_FILAS)
        # Checkpoint parcial
        if (i + 1) % CHECKPOINT_CADA == 0:
            df_es[col] = traducciones + list(df_es[col].iloc[i+1:])
            df_es.to_csv(OUTPUT_PATH + '.partial', index=False)
    df_es[col] = traducciones

print('\n✓ Traducción completada')


Traduciendo columna "text"…


text: 100%|██████████| 300/300 [01:48<00:00,  2.75it/s]



Traduciendo columna "explanation"…


explanation: 100%|██████████| 300/300 [01:24<00:00,  3.53it/s]



Traduciendo columna "context"…


context: 100%|██████████| 300/300 [01:24<00:00,  3.53it/s]


✓ Traducción completada


## 4. Verificación y guardado

In [6]:
import os

# Renombrar columna 'text' → 'descripcion' para compatibilidad con el pipeline
df_es = df_es.rename(columns={'text': 'descripcion'})

print('Columnas finales:', df_es.columns.tolist())
print()
print('Ejemplo — texto original vs traducido:')
print(f"  Original:  {df['text'].iloc[0][:120]}…")
print(f"  Traducido: {df_es['descripcion'].iloc[0][:120]}…")
print()
print('Distribución de etiquetas:')
print(df_es['etiqueta'].value_counts())

df_es.to_csv(OUTPUT_PATH, index=False)
print(f'\n✓ Dataset traducido guardado en: {OUTPUT_PATH}')
print(f'   Tamaño: {os.path.getsize(OUTPUT_PATH) / 1024:.1f} KB')

# Eliminar checkpoint parcial si existe
if os.path.exists(OUTPUT_PATH + '.partial'):
    os.remove(OUTPUT_PATH + '.partial')
    print('   Checkpoint parcial eliminado')

Columnas finales: ['descripcion', 'violation', 'category', 'severity', 'articles', 'explanation', 'context', 'split', 'ambiguity', 'etiqueta']

Ejemplo — texto original vs traducido:
  Original:  A smart city deploys an AI system for traffic light optimization that includes a public dashboard explaining the system'…
  Traducido: Una ciudad inteligente implementa un sistema de inteligencia artificial para la optimización del semáforo que incluye un…

Distribución de etiquetas:
etiqueta
inaceptable        131
riesgo_minimo      103
alto_riesgo         46
riesgo_limitado     20
Name: count, dtype: int64

✓ Dataset traducido guardado en: data/eu_ai_act_flagged_es.csv
   Tamaño: 219.3 KB
   Checkpoint parcial eliminado


## 5. Vista previa del dataset traducido

In [7]:
df_es[['descripcion', 'etiqueta', 'category', 'severity', 'context']].head(10)

,descripcion,etiqueta,category,severity,context
0,Una ciudad inteligente implementa un sistema d...,riesgo_minimo,transparency,none,ciudad inteligente
1,Un chatbot de servicio al cliente utiliza el r...,riesgo_limitado,risk_management,borderline,servicio al cliente
2,Un sistema de inteligencia artificial utilizad...,alto_riesgo,accuracy_robustness,high,seguro
3,Una plataforma de comercio electrónico utiliza...,inaceptable,data_governance,critical,comercio electrónico
4,Un sistema de inteligencia artificial de ciuda...,inaceptable,high_risk_systems,critical,ciudad inteligente
5,Un sistema de inteligencia artificial utilizad...,inaceptable,prohibited_practices,critical,contratación
6,Una plataforma de redes sociales utiliza IA pa...,riesgo_minimo,transparency,none,redes sociales
7,Una plataforma de comercio electrónico utiliza...,alto_riesgo,record_keeping,high,comercio electrónico
8,Una herramienta de inteligencia artificial de ...,riesgo_minimo,risk_management,none,servicios legales
9,Un chatbot de servicio al cliente que genera r...,inaceptable,transparency,critical,servicio al cliente
